In [1]:
import pandas as pd
import numpy as np
import json
import os

In [4]:
data_dir = "../data/onet"
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

print("Loading raw O*NET data...")
occupations_df = pd.read_csv(os.path.join(data_dir, "Occupation Data.txt"), sep='\t')
skills_df = pd.read_csv(os.path.join(data_dir, "Skills.txt"), sep='\t')
tech_skills_df = pd.read_csv(os.path.join(data_dir, "Technology Skills.txt"), sep='\t')
alt_titles_df = pd.read_csv(os.path.join(data_dir, "Alternate Titles.txt"), sep='\t')

Loading raw O*NET data...


In [ ]:
print("\n--- Starting Data Cleaning ---")

skills_df = skills_df[skills_df['Recommend Suppress'] != 'Y'].copy()

skills_df = skills_df[skills_df['Scale ID'] == 'IM'].copy()

tech_skills_df['Example'] = tech_skills_df['Example'].astype(str).str.strip()
tech_skills_df.dropna(subset=['Example'], inplace=True)
alt_titles_df['Alternate Title'] = alt_titles_df['Alternate Title'].astype(str).str.strip()
alt_titles_df.dropna(subset=['Alternate Title'], inplace=True)
occupations_df['Title'] = occupations_df['Title'].astype(str).str.strip()
occupations_df['Description'] = occupations_df['Description'].astype(str).str.strip()
print("Cleaning Complete.")


--- Starting Data Cleaning ---
Cleaning Complete.


In [ ]:

print("\n--- Starting Data Transformation ---")

print("Building skills matrix...")

skills_matrix = skills_df.pivot(index='O*NET-SOC Code', columns='Element Name', values='Data Value')

skills_matrix = skills_matrix / 5.0 

skills_matrix.fillna(0, inplace=True)
skills_matrix.reset_index(inplace=True)

skills_matrix.to_parquet(os.path.join(processed_dir, 'occupation_skills.parquet'), index=False)




occupations_out = occupations_df[['O*NET-SOC Code', 'Title', 'Description']]
occupations_out.to_parquet(os.path.join(processed_dir, 'occupations.parquet'), index=False)

tech_skills_out = tech_skills_df[['O*NET-SOC Code', 'Example', 'Commodity Title', 'Hot Technology']]
tech_skills_out.to_parquet(os.path.join(processed_dir, 'technology_skills.parquet'), index=False)

alt_titles_out = alt_titles_df[['O*NET-SOC Code', 'Alternate Title']]
alt_titles_out.to_parquet(os.path.join(processed_dir, 'alternate_titles.parquet'), index=False)

print("Extracting canonical skills...")
abstract_skills = list(skills_matrix.columns)
abstract_skills.remove('O*NET-SOC Code')
tech_skills = list(tech_skills_df['Example'].unique())

canonical_skills = sorted(list(set(abstract_skills + tech_skills)))
with open(os.path.join(processed_dir, 'canonical_skills.json'), 'w') as f:
    json.dump(canonical_skills, f, indent=4)
print("\n✅ Preprocessing Complete! All files successfully saved to data/processed/")



--- Starting Data Transformation ---
Building skills matrix...
Extracting canonical skills...

✅ Preprocessing Complete! All files successfully saved to data/processed/
